# Build a blocklist from seeds

Build a list of NSFW Wikipedia categories to block starting from a set of seeds.

This notebook requires the category index dataset to have been built previously by running `build_category_index.ipynb`.

In [1]:
from pathlib import Path

import pandas as pd

from wikipedia_utils.utils import display_pd
from wikipedia_utils import category

`DATA_DIR` should be set to the dir where the category index files were written.

In [2]:
DATA_DIR = Path("category_data")
SEED_FILE = "moderation_category_seeds.yml"
NEW_BLOCKLIST_CSV = DATA_DIR / "blocklist_cats_new.csv"
CURRENT_BLOCKLIST_CSV = DATA_DIR / "blocklist_cats_prev.csv"

First copy the current blocklist to the data dir.

In [3]:
! cp blocklist_cats.csv $CURRENT_BLOCKLIST_CSV

## Load the seed file

The final list of categories to block is built by expanding on a manually curated set of seeds.
The seeds are defined in a YAML file and consist of inclusions and exclusions, either exact category names and regexes, for each topic area.
These seeds have been chosen by interactively exploring Wikipedia categories relevant to the topic areas of interest using the tools described in `explore_categories.ipynb`.

The seed file has the following format. For each topic listing, all keys except for `topic` are optional.
```yaml
blocklist:
  - topic: <topic_key>
    seed_categories:
      - <exact category name>
    seed_re:
      - <regex to match category names>
    ignore_categories:
      - <exact category name>
    ignore_re:
      - <regex to match category names>
    max_level: <max depth to search category digraph>
```

In [4]:
seed_dict = category.load_blocklist_seeds(SEED_FILE)

We can view the seed list as a table with columns for each NSFW topic.

In [5]:
display_pd(pd.DataFrame(seed_dict).drop("topic"))

,profanity,pejoratives,cruelty,child_abuse,sexually_explicit,hateful_ideologies
seed_categories,[Profanity],NaN,[Torture],[Child abuse],"[Sex manuals, Incest, Sexual emotions, Prostitution, Sexuality-related lists, Machine sex, Orgasm, Sexual fetishism, Pornography, Paraphilias, Sexual slang, Personal lubricants, Sexual fantasies, Sex toys, Sexual violence, Sexual acts]","[White nationalism, Religious fanaticism]"
ignore_categories,[Works about profanity],[Barbarians],"[Castrated people, Operation Condor, Crucifixion, Amputation, Fictional torturers, Works about torture, Guantanamo Bay detention camp, People of the Dirty War]","[Child abuse law, Child abuse-related organizations, Anti-pedophile activism, Child labour, Displacement of indigenous children, School bullying, QAnon, Works about child abuse, Feral children, Fiction about child murder, Filicides, Fictional murderers of children, Child sacrifice]","[One Thousand and One Nights, Aphrodite, Game of Thrones, Erotic Liquid Culture members, Swedish Erotica members]",[Works about apartheid]
seed_re,NaN,"[\bslurs, \bpejorative]",NaN,NaN,"[\bpornograph, \berotic]",NaN
ignore_re,NaN,[\beskimo],"[\banti-torture\b, \btorture victims?\b, \bhuman rights activ]",NaN,"[\banti\W(child )?porn, \blaw.+\bporn, \bporn.+\blaw, \blaw.+\bprostitut, \bprostitut.+\blaw, \banti\Wprostitut, \bsex comedy\b, \bmytholog, \bNew Pornographers\b]",NaN
max_level,NaN,4,4,NaN,NaN,4


## Build blocklists for individual topics

To build a list of categories that will be used as a blocklist:

1. Identify all categories matched by the seeds, either exact category names or matching regexes. If seeds match both a subcategory and its parent, keep only the parent to ensure a cleaner level structure.
2. For each of these, identify all their subcategories, ignoring specified exact category names or matching regexes.
3. Repeat step 2 for all newly identified subcategories, until the depth cutoff is reached, or no more subcategories are available.

We can do this for individual topic specs included in the seed file. This is useful for verifying the final list of categories that will be included in the blocklist.

In [6]:
ci = category.CategoryIndex("category_data")

Individual blocklists for each category should be manually inspected to make sure they cover the desired set of topics.

In [7]:
%%time

ci.build_category_list_for_topic(seed_dict["profanity"])

CPU times: user 1.05 s, sys: 70.7 ms, total: 1.12 s
Wall time: 1.13 s


,name,seed,parent,level,topic
0,Profanity,Profanity,None,0,profanity
1,Profanity by language,Profanity,Profanity,1,profanity
2,Cantonese profanity,Profanity,Profanity by language,2,profanity
3,English profanity,Profanity,Profanity by language,2,profanity
4,Finnish profanity,Profanity,Profanity by language,2,profanity
5,French profanity,Profanity,Profanity by language,2,profanity
6,German profanity,Profanity,Profanity by language,2,profanity
7,Polish profanity,Profanity,Profanity by language,2,profanity
8,Russian profanity,Profanity,Profanity by language,2,profanity
9,Spanish profanity,Profanity,Profanity by language,2,profanity


## Build full blocklist

Finally, we build the full blocklist for the entire seed list.
This is done using the same method as above, across all topics, removing any duplicates.
The result is written directly to a CSV file.

In [8]:
%%time

ci.build_full_category_list(seed_dict, NEW_BLOCKLIST_CSV)

CPU times: user 17.2 s, sys: 576 ms, total: 17.8 s
Wall time: 17.8 s


Take a look at the generated blocklist of categories.

In [9]:
blocklist_cats = pd.read_csv(NEW_BLOCKLIST_CSV)

In [10]:
len(blocklist_cats)

5091

Compute some info for each topic:

In [11]:
blocklist_cats.groupby("topic").agg(
    num_categories=pd.NamedAgg("name", "size"),
    max_level=pd.NamedAgg("level", "max")
)

,num_categories,max_level
topic,,
child_abuse,499,9
cruelty,803,4
hateful_ideologies,947,4
pejoratives,224,4
profanity,10,2
sexually_explicit,2608,10


__NSFW content warning!__

Take a look at some of the categories

In [12]:
blocklist_cats.sample(20)

,name,seed,parent,level,topic
4967,United Confederate Veterans,White nationalism,Neo-Confederate organizations,4,hateful_ideologies
1926,Pornographic women's magazines,Erotic,Women's erotica and pornography,2,sexually_explicit
4454,People of the War in Iraq (2013–2017),Religious fanaticism,War in Iraq (2013–2017),4,hateful_ideologies
524,Harrods,Torture,Sexual abuse cover-ups,3,cruelty
111,Haredi Judaism in Israel,Pejorative,Haredi Judaism by country,4,pejoratives
3372,Swiss people convicted of rape,Sexual violence,People convicted of rape by nationality,5,sexually_explicit
4080,Fruits Basket character redirects,Erotic,Fruits Basket characters,7,sexually_explicit
957,Sexual harassment in India,Torture,Sexual harassment by country,4,cruelty
1564,Zoophilia,Paraphilias,Paraphilias,1,sexually_explicit
305,Sexual abuse in the military,Torture,Sexual abuse,2,cruelty


## Compare against previous blocklist

When we recompute the blocklist using a new category dump, we will want to see what changed relative to the previous version.

In [13]:
prev_bl_cats = pd.read_csv(CURRENT_BLOCKLIST_CSV)
new_bl_cats = pd.read_csv(NEW_BLOCKLIST_CSV)

How do the two lists differ?

In [14]:
bl_cats_comb = pd.concat(
    [prev_bl_cats.assign(type="previous"), new_bl_cats.assign(type="new")],
    ignore_index=True,
)

In [15]:
bl_cats_comb["type"].value_counts()

type
new         5091
previous    4969
Name: count, dtype: int64

In [16]:
(
    bl_cats_comb
    .groupby(["type", "topic"])
    .agg(num_categories=pd.NamedAgg("name", "size"), max_level=pd.NamedAgg("level", "max"))
    .unstack(level=0)
)

num_categories          max_level         
type                          new previous       new previous
topic                                                        
child_abuse                   499      495         9        9
cruelty                       803      767         4        4
hateful_ideologies            947      922         4        4
pejoratives                   224      175         4        4
profanity                      10       10         2        2
sexually_explicit            2608     2600        10       10

Compute a diff between the two lists.

In [17]:
bl_diff = category.compare_categories_lists(prev_bl_cats, new_bl_cats)

Some categories have been added to or removed from topics, or else their details (seed/parent/level) have changed.
In some cases, a category may move from one topic to another, but remains on the blocklist.
These changes can be due to edits made to Wikipedia, or updates we made to our seed list.

In [18]:
bl_diff["diff"].value_counts().sort_index()

diff
added      238
changed     25
moved       48
removed    116
Name: count, dtype: int64

__NSFW content warning!__

List the categories that differ. Categories that have moved to a different topic are shown in blue.

In [19]:
display_pd(category.style_diff(bl_diff))

,name,seed_previous,parent_previous,level_previous,topic,seed_new,parent_new,level_new,diff
2091,Ghislaine Maxwell,--,--,--,child_abuse,Child abuse,Jeffrey Epstein,5,moved
2756,Killing of JonBenét Ramsey,--,--,--,child_abuse,Child abuse,Murdered American children,5,added
3674,Properties of Jeffrey Epstein,--,--,--,child_abuse,Child abuse,Jeffrey Epstein,5,moved
5091,Works about Jeffrey Epstein,--,--,--,child_abuse,Child abuse,Jeffrey Epstein,5,moved
542,Blinded people,Torture,Mutilation,2,cruelty,Torture,Corporal punishments,2,changed
175,Activists against circumcision,Torture,Genital modification and mutilation,3,cruelty,Torture,Genital integrity activists,4,changed
176,Activists against female genital mutilation,Torture,Genital modification and mutilation,3,cruelty,Torture,Female genital mutilation,3,changed
462,BDSM equipment,Torture,Instruments of torture,3,cruelty,Torture,Physical restraint,3,changed
1996,Gamergate,Torture,Sexual harassment,3,cruelty,Torture,Sexual harassment journalism,4,changed
3408,Pillories,Torture,European instruments of torture,4,cruelty,Torture,Corporal punishments,2,changed


For the categories that have moved to a different topic, where did they end up?

In [20]:
category.style_diff(
    bl_diff.query("diff == 'moved'").sort_values(["name", "level_previous"])
)

,name,seed_previous,parent_previous,level_previous,topic,seed_new,parent_new,level_new,diff
550,Bondage (BDSM),Sexual acts,BDSM activities,2,sexually_explicit,--,--,--,moved
549,Bondage (BDSM),--,--,--,cruelty,Torture,Physical restraint,3,moved
552,Bondage artists,Sexual acts,Bondage (BDSM),3,sexually_explicit,--,--,--,moved
551,Bondage artists,--,--,--,cruelty,Torture,Bondage (BDSM),4,moved
554,Bondage magazines,Sexual fetishism,Fetish magazines,2,sexually_explicit,--,--,--,moved
553,Bondage magazines,--,--,--,cruelty,Torture,Bondage (BDSM),4,moved
556,Bondage models,Sexual acts,Bondage (BDSM),3,sexually_explicit,--,--,--,moved
555,Bondage models,--,--,--,cruelty,Torture,Bondage (BDSM),4,moved
558,Bondage pornography,Sexual acts,Bondage (BDSM),3,sexually_explicit,--,--,--,moved
557,Bondage pornography,--,--,--,cruelty,Torture,Bondage (BDSM),4,moved
